# Tydzień 3: Regresja – przygotowanie danych `setup()` i porównanie modeli `compare_models()` ⚙️

| | |
|---|---|
| **Rozdział książki** | 2. *Regression* – podrozdziały: *Initializing the PyCaret Environment*, *Comparing Regression Models* |
| **Czas zajęć** | 90 minut |
| **Dane** | `insurance` – koszty ubezpieczenia zdrowotnego (wbudowane w PyCaret) |
| **Nowe funkcje PyCaret** | `setup()`, `get_config()`, `models()`, `compare_models()`, `pull()` |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. rozumieć, po co dzielimy dane na **zbiór treningowy i testowy** (*train-test split*),
2. wiedzieć, czym są **kodowanie one-hot**, **normalizacja** i **transformacja** zmiennych,
3. rozumieć ideę **walidacji krzyżowej** (*cross-validation*),
4. umieć porównać ok. 20 modeli regresji jednym poleceniem i odczytać metryki **MAE**, **RMSE** i **R²**.

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–25 min | 📚 Teoria: przygotowanie danych, walidacja, metryki |
| 25–65 min | 👣 Krok po kroku: `setup()` i `compare_models()` |
| 65–85 min | 🧑‍💻 Ćwiczenie samodzielne – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Przygotowanie danych = „mise en place” w kuchni 🍳
Kucharz przed gotowaniem kroi warzywa, odmierza przyprawy i ustawia wszystko pod ręką. Funkcja `setup()` robi to samo z danymi:
tworzy **potok przetwarzania** (*preprocessing pipeline*) – listę kroków, które przygotowują dane dla modelu.
Najważniejsze kroki:

### 1. Podział na zbiór treningowy i testowy (*Train-Test Split*) ✂️
Nauczyciel nie daje na egzaminie **dokładnie tych samych zadań**, które były na ćwiczeniach – chce sprawdzić, czy uczeń
naprawdę rozumie, a nie tylko pamięta. Dlatego dzielimy dane:
- **zbiór treningowy** (*train set*, np. 80%) – na nim model się uczy,
- **zbiór testowy** (*test set*, np. 20%) – „egzamin” na danych, których model nigdy nie widział.

### 2. Kodowanie one-hot (*One-Hot Encoding*) 💡
Modele rozumieją tylko liczby, a `region` to tekst (`southwest`, `northeast`, …). Kodowanie one-hot zamienia jedną kolumnę
na kilka kolumn 0/1 – jak **rząd włączników światła**, w którym świeci się dokładnie jedna żarówka:

| region | region_southwest | region_southeast | region_northwest | region_northeast |
|---|---|---|---|---|
| southwest | **1** | 0 | 0 | 0 |
| northeast | 0 | 0 | 0 | **1** |

### 3. Normalizacja (*Normalization*) 📏
`age` ma wartości 18–64, `bmi` 15–53, a `children` 0–5. Niektóre modele „myślą”, że większe liczby są ważniejsze.
To tak, jakby porównywać ceny podane raz w złotówkach, a raz w groszach. Normalizacja (tzw. *z-score*) sprowadza
każdą cechę do wspólnej skali: średnia = 0, odchylenie standardowe = 1.

$$z = \frac{x - \text{średnia}}{\text{odchylenie standardowe}}$$

### 4. Transformacja (*Transformation*) 🔧
W zeszłym tygodniu widzieliśmy, że niektóre rozkłady są **skośne**. Transformacja „prostuje” je tak, by bardziej
przypominały symetryczny „dzwon” (rozkład normalny), co pomaga części modeli.

## Walidacja krzyżowa (*K-Fold Cross-Validation*) 🔁
Jeden sprawdzian może być „pechowy” (za trudne pytania). Lepiej napisać **10 sprawdzianów** z różnych części materiału
i policzyć średnią. Tak działa 10-krotna walidacja krzyżowa: zbiór treningowy dzielimy na 10 części (*folds*);
model uczy się na 9, sprawdza się na 1 – i tak 10 razy, za każdym razem na innej części. Wynik = średnia z 10 prób.

## Metryki regresji – „jak bardzo się mylimy?” 🎯
| Metryka | Co znaczy? | Lepiej, gdy… |
|---|---|---|
| **MAE** (*Mean Absolute Error*) | średni błąd w dolarach („średnio mylimy się o X $”) | mniejsza |
| **RMSE** (*Root Mean Squared Error*) | jak MAE, ale **mocniej karze duże pomyłki** | mniejsza |
| **R²** (*R-squared*) | jaką część zmienności kosztów model wyjaśnia (1 = ideał, 0 = nic) | większa (bliżej 1) |

W tym zbiorze skupimy się na **RMSE** (tak jak w książce).

---
# 👣 Część 2: Krok po kroku

### Krok 1: Importy i dane

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# pandas i matplotlib – tabele i wykresy
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# get_data – wczytywanie zbiorów; "import *" importuje WSZYSTKIE funkcje modułu regresji PyCaret
from pycaret.datasets import get_data
from pycaret.regression import *

In [ ]:
# Wczytujemy dane o kosztach ubezpieczenia (te same co tydzień temu)
data = get_data('insurance')

In [ ]:
# Wskazujemy, które cechy są liczbowe, a które kategoryczne (bez 'charges' – to zmienna celu!)
numeric = ['age', 'bmi', 'children']
categorical = ['smoker', 'sex', 'region']

### Krok 2: Inicjalizacja środowiska – `setup()` 🏗️
To najważniejsza funkcja PyCaret. Każdy parametr opisaliśmy w komentarzu.
Po uruchomieniu zobaczysz tabelę z ustawieniami – omówimy ją poniżej.

In [ ]:
reg = setup(
    data=data,                         # nasza tabela z danymi
    target='charges',                  # zmienna celu (target) – co przewidujemy
    train_size=0.8,                    # 80% danych do treningu, 20% do testu (train-test split)
    session_id=7402,                   # "ziarno losowości" (seed) – dzięki niemu wszyscy dostaniemy te same wyniki
    numeric_features=numeric,          # cechy liczbowe
    categorical_features=categorical,  # cechy kategoryczne
    transformation=True,               # transformacja – "prostowanie" skośnych rozkładów
    normalize=True,                    # normalizacja (z-score): średnia 0, odchylenie 1
)

📖 **Jak czytać tabelę z `setup()`?** Najważniejsze wiersze:
- `Target` – zmienna celu (`charges`), `Target type` – typ zadania (*Regression*),
- `Original data shape` – rozmiar danych przed przygotowaniem: (1338, 7),
- `Transformed train set shape` / `Transformed test set shape` – rozmiar zbiorów po przygotowaniu
  (kolumn jest więcej, bo `region` zamienił się na 4 kolumny 0/1),
- `Numeric features` / `Categorical features` – liczba cech każdego typu,
- `Imputation type` – sposób uzupełniania braków danych (u nas braków nie ma, ale PyCaret jest przygotowany),
- `Transformation` / `Normalize` – włączone kroki przygotowania (metody: `yeo-johnson` i `zscore`),
- `Fold Generator` = `KFold`, `Fold Number` = 10 – walidacja krzyżowa 10-krotna.

### Krok 3: Zaglądamy do przygotowanych danych 🔬
Funkcja `get_config()` pozwala podejrzeć „wnętrze” eksperymentu PyCaret.

In [ ]:
# Rozmiary zbioru treningowego i testowego (wiersze, kolumny)
print("Zbiór treningowy:", get_config('X_train').shape)
print("Zbiór testowy:   ", get_config('X_test').shape)

In [ ]:
# Dane treningowe PRZED przygotowaniem (oryginalne wartości)
get_config('X_train').head()

In [ ]:
# Dane treningowe PO przygotowaniu (transformed) – to "widzi" model
get_config('X_train_transformed').head()

🔎 **Co się zmieniło?**
- `age`, `bmi`, `children` mają teraz „dziwne” wartości wokół 0 – to efekt transformacji i normalizacji,
- `sex` i `smoker` (tylko 2 kategorie) zamieniły się w liczby,
- `region` (4 kategorie) zamienił się w **4 kolumny** `region_...` (kodowanie one-hot).

In [ ]:
# Lista kroków potoku przetwarzania (pipeline) – nazwy kolejnych "stanowisk w kuchni"
[name for name, step in get_config('pipeline').steps]

### Krok 4: Jakie modele mamy do dyspozycji?

In [ ]:
# models() zwraca tabelę dostępnych modeli: ID (skrót używany w kodzie), pełną nazwę i bibliotekę źródłową
models()

### Krok 5: Porównanie wszystkich modeli – `compare_models()` 🏁
PyCaret wytrenuje **każdy** model z listy, oceni go 10-krotną walidacją krzyżową i ułoży ranking.
⏳ To potrwa **ok. 1–3 minuty** – w tym czasie możesz obserwować pasek postępu. Żółte tło w tabeli oznacza najlepszy wynik w kolumnie.

In [ ]:
# sort='RMSE' – ranking według RMSE (od najmniejszego błędu). Najlepszy model trafia do zmiennej best.
best = compare_models(sort='RMSE')

In [ ]:
# Wypisujemy najlepszy model wraz z jego ustawieniami
print(best)

In [ ]:
# pull() zwraca ostatnio wyświetloną tabelę wyników jako DataFrame – możemy ją dalej analizować
results = pull()
results.head()

In [ ]:
# Wykres słupkowy RMSE dla 10 najlepszych modeli (krótszy słupek = lepszy model)
results.head(10).plot(kind='barh', x='Model', y='RMSE', figsize=(8, 5))
plt.gca().invert_yaxis()   # odwracamy oś, żeby najlepszy model był na górze
plt.show()

## 🧾 Jak interpretować wyniki?
- Najlepiej wypadają modele oparte na **drzewach decyzyjnych** (*Gradient Boosting*, *CatBoost*, *Random Forest*):
  najlepszy, **Gradient Boosting Regressor**, ma RMSE ok. **4 750 $** i R² ok. **0,84** – wyjaśnia ok. 84% zmienności kosztów.
- **Regresja liniowa** (`lr`) ma RMSE ok. **6 100 $** i R² ok. **0,73** – jest gorsza, bo zależności w danych nie są proste
  (pamiętasz „trzy pasma” i interakcję palenie × BMI z zeszłego tygodnia?).
- Na samym dole jest **Dummy Regressor** – model „na ślepo”, który zawsze przewiduje średnią. To **punkt odniesienia**
  (*baseline*): każdy sensowny model musi być od niego wyraźnie lepszy.
- Kolumna `TT (Sec)` to czas treningu – czasem warto wybrać nieco słabszy, ale dużo szybszy model.

💡 Wyniki mogą się minimalnie różnić od książki (inna wersja PyCaret), ale ranking jest bardzo podobny.

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie:** Sprawdź, czy przygotowanie danych (normalizacja i transformacja) ma znaczenie.

- **a)** Uruchom nowy `setup()` na tych samych danych, ale **tylko** z parametrami `data`, `target` i `session_id=7402`
  (bez normalizacji i transformacji).
- **b)** Porównaj tylko 4 modele: regresję liniową, drzewo decyzyjne, las losowy i gradient boosting
  (ID: `'lr'`, `'dt'`, `'rf'`, `'gbr'`), sortując według `'MAE'`.
- **c)** Czy najlepszy model się zmienił? Czy wyniki modeli drzewiastych bardzo się różnią od poprzednich?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Nowy `setup()` po prostu **zastępuje** poprzedni eksperyment.
- Domyślnie `normalize=False` i `transformation=False`, więc wystarczy ich nie podawać.
- Do wyboru modeli służy parametr `include`, np. `compare_models(include=['lr', 'dt'], sort='MAE')`.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Czy ranking zależy od metryki?
Wróć do pełnego przygotowania danych (z normalizacją i transformacją, jak w kroku 2), a następnie:

- **a)** porównaj wszystkie modele **5-krotną** walidacją krzyżową (szybciej niż 10-krotna), sortując według **MAE**,
- **b)** zapisz **3 najlepsze** modele w zmiennej `top3` i wypisz ich nazwy,
- **c)** porównaj ranking z tabelą z kroku 5 (sortowanie po RMSE). Czy kolejność modeli jest taka sama? Dlaczego może się różnić?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Ponownie uruchom `setup(...)` z kroku 2 (możesz dodać `verbose=False`).
- `compare_models()` ma parametry `fold=` (liczba części walidacji) i `n_select=` (ile najlepszych modeli zwrócić – wtedy wynikiem jest **lista**).
- Nazwę klasy modelu wypiszesz tak: `type(model).__name__`; po liście przejdziesz pętlą `for model in top3:`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Inżynieria cech – pomóżmy regresji liniowej
W tygodniu 2 odkryliśmy **interakcję**: koszty bardzo rosną u osób, które **palą i mają otyłość** (BMI ≥ 30).
Regresja liniowa sama takiej zależności „nie zauważy”. Możemy jej pomóc, tworząc nową cechę – to tzw. **inżynieria cech** (*feature engineering*).

- **a)** Zbuduj model regresji liniowej (`'lr'`) w obecnym środowisku i zanotuj jego średnie RMSE i R².
- **b)** Utwórz kopię danych `data2` z nową kolumną `smoker_obese` = 1, gdy osoba **pali i** ma BMI ≥ 30, w przeciwnym razie 0.
- **c)** Uruchom `setup()` na `data2` (te same ustawienia) i ponownie zbuduj `'lr'`. Jak zmieniło się RMSE?
- **d)** Porównaj wynik z najlepszym modelem z kroku 5 (*Gradient Boosting*, RMSE ok. 4 750 $). Co z tego wynika?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Wynik ostatniego modelu odczytasz przez `pull()`; średnie to wiersz `'Mean'`: `pull().loc['Mean', ['RMSE', 'R2']]`.
- Kopia tabeli: `data2 = data.copy()`.
- Warunek „pali i otyłość”: `(data2['smoker'] == 'yes') & (data2['bmi'] >= 30)`, a zamiana True/False na 1/0: `.astype(int)`.
- W `setup()` dla `data2` nie podawaj list `numeric_features` / `categorical_features` (nowej kolumny w nich nie ma) – PyCaret sam rozpozna typy.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- `setup()` tworzy **potok przygotowania danych**: podział train/test, kodowanie one-hot, normalizacja, transformacja.
- `get_config()` pozwala podejrzeć dane przed i po przygotowaniu.
- `compare_models()` trenuje i porównuje wszystkie modele przy pomocy **walidacji krzyżowej**.
- Metryki: **MAE** i **RMSE** (im mniej, tym lepiej), **R²** (im bliżej 1, tym lepiej).
- W naszych danych wygrywa **Gradient Boosting Regressor** (`gbr`).

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| podział na zbiór treningowy i testowy | train-test split | oddzielenie danych do nauki i do „egzaminu” |
| potok przetwarzania | (preprocessing) pipeline | lista kroków przygotowania danych |
| kodowanie one-hot | one-hot encoding | zamiana kategorii na kolumny 0/1 |
| normalizacja / standaryzacja | normalization / standardization (z-score) | sprowadzenie cech do wspólnej skali |
| transformacja | (power) transformation | „prostowanie” skośnych rozkładów |
| walidacja krzyżowa | cross-validation (k-fold) | wielokrotne sprawdzanie modelu na różnych częściach danych |
| fałda | fold | jedna z k części danych w walidacji krzyżowej |
| średni błąd bezwzględny | Mean Absolute Error (MAE) | średnia pomyłka w jednostkach celu |
| pierwiastek błędu średniokwadratowego | Root Mean Squared Error (RMSE) | błąd mocniej karzący duże pomyłki |
| współczynnik determinacji | R² (R-squared) | jaka część zmienności jest wyjaśniona |
| ziarno losowości | random seed (`session_id`) | zapewnia powtarzalność wyników |

➡️ **Za tydzień:** zbudujemy najlepszy model, **dostroimy** go (`tune_model`), narysujemy jego błędy i **zapiszemy** go do pliku.